In [1]:
import re
import emoji
import plotly.express as px
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Constants
MODEL_ID = "cardiffnlp/twitter-roberta-base-irony"
MODEL_REVISION = "3bf8f118bdf6b00c99658151ef10c9a0b9afd6bf"

c:\Users\Lukas\Master_AI\3rd_Semester\UE_AI_and_Visualization\UE\Project\.venv-irony\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset for preprocessing

# get dataset form HuggingFace
tweets = load_dataset("cardiffnlp/tweet_eval", "irony", split="test").to_pandas()
# Label tweets ironic and non ironic
tweets["reference_label"] = tweets["label"].map({
    0: "Non-ironic",
    1: "Ironic",
})

# tags to remove: #irony, #sarcasm, #not
explicit_tag = re.compile(r"(?<!\w)#(?:irony|sarcasm|not)\b", re.IGNORECASE)

# remove explicit tags from the text and store them in a new column
tweets["removed_tags"] = tweets["text"].map(
    lambda text: ", ".join(explicit_tag.findall(text))
)
tweets["edited_text"] = tweets["text"].map(
    lambda text: explicit_tag.sub("", text).strip()
)

tweets.groupby("reference_label")["removed_tags"].agg(
    total="size",
    with_removed_tags=lambda tags: tags.ne("").sum(),
)

,total,with_removed_tags
reference_label,,
Ironic,311,311
Non-ironic,473,113


In [3]:
# Resolve the Hub checkpoint explicitly, even if a local folder has the same name
checkpoint = snapshot_download(
    MODEL_ID,
    revision=MODEL_REVISION,
    allow_patterns=["config.json", "pytorch_model.bin", "merges.txt",
                    "vocab.json", "special_tokens_map.json"],
)

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.eval()

# Normalize tweets by replacing user mentions and URLs with nothing
def normalize_tweet(text):
    tokens = []
    for token in text.split():
        if token.startswith("@") and len(token) > 1:
            token = "@user"
        elif token.startswith("http"):
            token = "http"
        tokens.append(token)
    return " ".join(tokens)

# Compute irony scores for a list of texts in batches   
@torch.inference_mode()
def irony_scores(texts, batch_size=16):
    scores = []

    for start in range(0, len(texts), batch_size):
        batch = [
            normalize_tweet(text)
            for text in texts[start:start + batch_size]
        ]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        logits = model(**inputs).logits
        scores.extend(logits.softmax(dim=-1)[:, 1].cpu().tolist())

    return scores


# Compute irony scores for the original tweets
tweets["score_before"] = irony_scores(tweets["text"].tolist())

# Analyze the changes
changed = tweets.loc[tweets["removed_tags"].ne("")].copy()
changed["score_after"] = irony_scores(changed["edited_text"].tolist())
changed["score_delta"] = changed["score_after"] - changed["score_before"]

changed["class_flip"] = (
    changed["score_before"].ge(0.5) != changed["score_after"].ge(0.5)
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18502.25it/s]


In [4]:
# Plot the whole thing

if changed.empty:
    print("No hashtags found in the test split.")
else:
    fig = px.scatter(
        changed,
        x="score_before",
        y="score_after",
        color="reference_label",
        symbol="class_flip",
        opacity=0.65,
        category_orders={
            "reference_label": ["Ironic", "Non-ironic"],
            "class_flip": [False, True],
        },
        hover_data={
            "text": True,
            "edited_text": True,
            "removed_tags": True,
            "score_delta": ":.3f",
        },
        labels={
            "score_before": "irony-Score · original",
            "score_after": "irony-Score · without irony-hashtags",
            "reference_label": "reference label",
            "class_flip": "class flip",
            "text": "original text",
            "edited_text": "edited text",
            "removed_tags": "removed tags",
            "score_delta": "score delta",
        },
        template="plotly_white",
    )

    fig.add_shape(
        type="line",
        x0=0, y0=0, x1=1, y1=1,
        line=dict(color="gray", dash="dash"),
    )
    fig.add_vline(x=0.5, line_dash="dot")
    fig.add_hline(y=0.5, line_dash="dot")
    fig.update_xaxes(range=[0, 1], tick0=0, dtick=0.2, constrain="domain")
    fig.update_yaxes(
        range=[0, 1],
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    )
    legend_labels = {
        "Ironic, False": "Ironic",
        "Ironic, True": "Ironic · Change",
        "Non-ironic, False": "Non-ironic",
        "Non-ironic, True": "Non-ironic · Change",
    }
    fig.for_each_trace(lambda trace: trace.update(name=legend_labels[trace.name]))
    fig.update_xaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_yaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_layout(
        width=600,
        modebar_remove=["toImage", "select2d", "lasso2d", "zoomIn2d",
                        "zoomOut2d", "autoScale2d"],
        height=640,
        margin=dict(l=61, r=19, t=66, b=54),
        title=dict(
            text=f"Hashtag Removal · {len(changed)} Tweets",
            x=0,
            xref="paper",
            xanchor="left",
            y=0.985,
            yanchor="top",
            font=dict(size=21),
        ),
        legend=dict(
            orientation="h",
            x=0,
            xanchor="left",
            y=1.012,
            yanchor="bottom",
            title_text="",
            font=dict(size=11),
        ),
    )
    fig.show()

In [5]:
# naalyze effect of removing emojis from tweets
emoji_test = tweets[["text", "reference_label", "score_before"]].copy()

# extract emojis from text 
emoji_test["removed_emojis"] = emoji_test["text"].map(
    lambda text: " ".join(item["emoji"] for item in emoji.emoji_list(text))
)

# only keep tweets that actually contain emojis
emoji_test = emoji_test.loc[emoji_test["removed_emojis"].ne("")].copy()

# remove emojis from text and compute irony scores
emoji_test["without_emojis"] = emoji_test["text"].map(
    lambda text: emoji.replace_emoji(text, replace="").strip()
)

# compute irony scores for the tweets without emojis
emoji_test["score_after_emoji"] = irony_scores(
    emoji_test["without_emojis"].tolist()
)
emoji_test["delta_emoji"] = (
    emoji_test["score_after_emoji"] - emoji_test["score_before"]
)

# check for class flips
before_ironic = emoji_test["score_before"].ge(0.5)
after_ironic = emoji_test["score_after_emoji"].ge(0.5)

# indicate transitions
emoji_test["transition"] = "Kein Klassenwechsel"
emoji_test.loc[before_ironic & ~after_ironic, "transition"] = (
    "Ironisch → Nicht ironisch"
)
emoji_test.loc[~before_ironic & after_ironic, "transition"] = (
    "Nicht ironisch → Ironisch"
)

# indicate class flips
emoji_test["class_flip"] = before_ironic.ne(after_ironic)

print(f"{len(emoji_test)} von {len(tweets)} Tweets enthalten Emojis.")

emoji_test.groupby("reference_label").agg(
    tweets_mit_emojis=("text", "size"),
    median_score_aenderung=("delta_emoji", "median"),
    klassenwechsel=("class_flip", "sum"),
)

88 von 784 Tweets enthalten Emojis.


,tweets_mit_emojis,median_score_aenderung,klassenwechsel
reference_label,,,
Ironic,33,0.070080,6
Non-ironic,55,0.029399,8


In [6]:
# Plot the whole thing


if emoji_test.empty:
    print("No emojis found in the test split.")
else:
    fig = px.scatter(
        emoji_test,
        x="score_before",
        y="score_after_emoji",
        color="reference_label",
        symbol="class_flip",
        opacity=0.65,
        category_orders={
            "reference_label": ["Ironic", "Non-ironic"],
            "class_flip": [False, True],
        },
        hover_data={
            "text": True,
            "without_emojis": True,
            "removed_emojis": True,
            "delta_emoji": ":.3f",
        },
        labels={
            "score_before": "irony  · Original",
            "score_after_emoji": "irony-Score · without Emojis",
            "reference_label": "reference label",
            "class_flip": "class flip",
            "text": "original text",
            "without_emojis": "text without Emojis",
            "removed_emojis": "removed Emojis",
            "delta_emoji": "score delta",
        },
        template="plotly_white",
    )

    fig.add_shape(
        type="line",
        x0=0, y0=0, x1=1, y1=1,
        line=dict(color="gray", dash="dash"),
    )
    fig.add_vline(x=0.5, line_dash="dot")
    fig.add_hline(y=0.5, line_dash="dot")
    fig.update_xaxes(range=[0, 1], tick0=0, dtick=0.2, constrain="domain")
    fig.update_yaxes(
        range=[0, 1],
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    )
    legend_labels = {
        "Ironic, False": "Ironic",
        "Ironic, True": "Ironic · Change",
        "Non-ironic, False": "Non-ironic",
        "Non-ironic, True": "Non-ironic · Change",
    }
    fig.for_each_trace(lambda trace: trace.update(name=legend_labels[trace.name]))
    fig.update_xaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_yaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_layout(
        width=600,
        modebar_remove=["toImage", "select2d", "lasso2d", "zoomIn2d",
                        "zoomOut2d", "autoScale2d"],
        height=640,
        margin=dict(l=61, r=19, t=66, b=54),
        title=dict(
            text=f"Emoji Removal · {len(emoji_test)} Tweets",
            x=0,
            xref="paper",
            xanchor="left",
            y=0.985,
            yanchor="top",
            font=dict(size=21),
        ),
        legend=dict(
            orientation="h",
            x=0,
            xanchor="left",
            y=1.012,
            yanchor="bottom",
            title_text="",
            font=dict(size=11),
        ),
    )
    fig.show()